**Prerequisites:** This notebook requires additional packages for plotting and MCMC visualization. Install them with:
```
pip install matplotlib corner tqdm pandas
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import corner
from VegasAfterglow import Fitter, ParamDef, Scale

In [ ]:
######### 1. CONFIGURE MODEL & PREPARE DATA #########
fitter = Fitter(
    z=1.58,
    lumi_dist=3.364e28,
    jet="tophat",
    medium="wind",
)

# Define the frequency bands [Hz], filter/instrument labels, and light-curve files
band = [2.4e17, 4.84e14, 1.4e14]
labels = ["WXT", "r", "VT_R"]  # used by draw_fit() for legend display
lc_files = ["data/ep.csv", "data/r.csv", "data/vt-r.csv"]

# Add light curves at each frequency band
for nu, lbl, fname in zip(band, labels, lc_files):
    df = pd.read_csv(fname)
    fitter.add_flux_density(nu=nu, label=lbl, t=df["t"], f_nu=df["Fv_obs"], err=df["Fv_err"])  # All quantities in CGS units

# Add spectra at specific times
times = [3000]
spec_files = ["data/ep-spec.csv"]    

for t, fname in zip(times, spec_files):
    df = pd.read_csv(fname)
    fitter.add_spectrum(t=t, nu=df["nu"], f_nu=df["Fv_obs"], err=df["Fv_err"])  # All quantities in CGS units

######### 2. DEFINE PARAMETERS #########
# Parameter name, lower bound, upper bound, scale type, initial value (optional)
mc_params = [
    ParamDef("E_iso",    1e51,  1e54,  Scale.log,    1e53),    # Isotropic energy [erg]
    ParamDef("Gamma0",      5,  1000,  Scale.log,      20),    # Lorentz factor at the core
    ParamDef("theta_c",   0.0,   0.5,  Scale.linear,  0.3),    # Core half-opening angle [rad]
    ParamDef("theta_v",   0.0,   0.0,  Scale.fixed,   0.0),    # Viewing angle [rad]
    ParamDef("p",           2,     3,  Scale.linear,  2.3),    # Power law index
    ParamDef("eps_e",    1e-2,   0.3,  Scale.log,    0.05),    # Electron energy fraction
    ParamDef("eps_B",    1e-4,   0.3,  Scale.log,    0.03),    # Magnetic field energy fraction
    ParamDef("xi_e",     1e-3,   0.1,  Scale.log,    0.01),    # Electron acceleration efficiency
    ParamDef("A_star",   1e-3,    10,  Scale.log,    0.05),    # Wind parameter 
]

######### 3. RUN MCMC SAMPLING #########

# Run with emcee sampler
result = fitter.fit(
    mc_params,
    sampler="emcee",
    resolution=(0.1, 0.25, 10), # Grid resolution (phi, theta, t) - affects model accuracy
    nsteps=10000,              # Number of steps per walker 
    nburn=2000,                # Burn-in steps to discard
    top_k=10,                  # Number of best-fit parameters to return
)

# Alternative: Run with dynesty nested sampler (computes Bayesian evidence)
# result = fitter.fit(
#     mc_params,
#     sampler="dynesty",
#     resolution=(0.3, 0.3, 10),
#     nlive=500,
#     dlogz=0.1,
#     top_k=10,
# )

print(f"Total samples: {result.samples.shape[0]}")

# Top-K best-fit parameters with chi^2 (rank | chi^2 | param values)
print(result.summary())

# Persist EVERYTHING (constructor args, data, parameter defs, samples).
# Format is bilby-native HDF5 -- also openable directly via bilby.read_in_result().
fitter.save("vegas_mcmc_fit.h5")

# Reload later in a single line -- no Fitter reconfiguration, no MCMC re-run:
#
#   from VegasAfterglow import Fitter
#   fitter = Fitter.load("vegas_mcmc_fit.h5")
#   result = fitter.result
#   lc   = fitter.flux_density_grid(result.top_k_params[0], t_out, band).total
#   spec = fitter.flux_density_grid(result.top_k_params[0], times,  nu_out).total

In [ ]:
# Define time and frequency ranges for predictions
t_out = np.logspace(2, 9, 150)
nu_out = np.logspace(16, 20, 150)

# 68% credible bands from the posterior, with the median trajectory as the
# central line (median is always inside [lower, upper], the MAP need not be).
lc_band = fitter.flux_density_credible(t_out, band, ci=0.68, n_samples=100)
spec_band = fitter.flux_density_credible(times, nu_out, ci=0.68, n_samples=100)


In [ ]:
# Plot posterior-median light curves & spectra with 1σ credible bands.
def draw_bestfit(t, lc_band, nu, spec_band):
    fig = plt.figure(figsize=(4.5, 7.5))
    ax1 = fig.add_subplot(211)
    ax2 = fig.add_subplot(212)

    shift = [1, 1, 200]
    colors = ['blue', 'orange', 'green']

    for i, file, sft, c in zip(range(len(lc_files)), lc_files, shift, colors):
        df = pd.read_csv(file)
        ax1.errorbar(df["t"], df["Fv_obs"] * sft, df["Fv_err"] * sft,
                     fmt='o', markersize=4, label=file, color=c,
                     markeredgecolor='k', markeredgewidth=.4)
        ax1.plot(t, lc_band.median[i, :] * sft, color=c, lw=1)
        ax1.fill_between(t, lc_band.lower[i, :] * sft, lc_band.upper[i, :] * sft,
                         color=c, alpha=0.2, lw=0)

    ax1.set_xscale('log')
    ax1.set_yscale('log')
    ax1.set_xlabel('t [s]')
    ax1.set_ylabel(r'$F_\nu$ [erg/cm$^2$/s/Hz]')
    ax1.legend()

    for i, file, sft, c in zip(range(len(spec_files)), spec_files, shift, colors):
        df = pd.read_csv(file)
        ax2.errorbar(df["nu"], df["Fv_obs"] * sft, df["Fv_err"] * sft,
                     fmt='o', markersize=4, label=file, color=c,
                     markeredgecolor='k', markeredgewidth=.4)
        ax2.plot(nu, spec_band.median[:, i] * sft, color=c, lw=1)
        ax2.fill_between(nu, spec_band.lower[:, i] * sft, spec_band.upper[:, i] * sft,
                         color=c, alpha=0.2, lw=0)

    ax2.set_xscale('log')
    ax2.set_yscale('log')
    ax2.set_xlabel(r'$\nu$ [Hz]')
    ax2.set_ylabel(r'$F_\nu$ [erg/cm$^2$/s/Hz]')
    ax2.legend()
    plt.tight_layout()

draw_bestfit(t_out, lc_band, nu_out, spec_band)


In [ ]:
######### 4b. ONE-CALL DIAGNOSTIC PLOT #########
# Same plot as above (light-curve data + best-fit model) in a single call,
# plus a bottom panel showing how ν_a, ν_m, ν_c evolve in the observer frame.

fig, (ax_top, ax_bot) = fitter.draw_fit()  # defaults: top_k_params[0]
plt.tight_layout()
plt.show()

In [ ]:
######### 5. VISUALIZATION #########

# Flatten samples (for nested sampling, samples are weighted -- this is simplified)
flat_chain = result.samples.reshape(-1, result.samples.shape[-1])

fig = corner.corner(
    flat_chain,
    labels=result.latex_labels,
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True,
    truths=np.median(flat_chain, axis=0),
    truth_color='red',
    smooth=1,
    bins=30,
    plot_datapoints=False,
    fill_contours=True,
    levels=[0.16, 0.5, 0.68],
    color='k',
)
fig.savefig("corner_plot.png", dpi=600, bbox_inches='tight')

# For a weighted corner plot, use bilby's built-in method:
# result.bilby_result.plot_corner(filename="corner_bilby.png")